 ```text
START
  ↓
READ ESSAY + 4-SECTION RUBRIC
  ↓
CHECK 1: Presentation (20%)
  → Evaluate: Overall presentation and quality of communication
  → grade_pres, feedback_pres, evidence_pres
  ↓
CHECK 2: Reflection Methodology (20%)
  → 2.1: Evidence of continuous reflection during study?
  → 2.2: Used conventional reflection model? (Gibbs, Kolb, etc.)
        ↓
    If YES → 2.2A: Evaluate correctness & degree of application
             → grade_refl, feedback_refl, evidence_refl
        ↓
    If NO  → 2.2B: Evaluate reflective language & writing style
             → grade_refl, feedback_refl, evidence_refl
  ↓
CHECK 3: Breadth (30%)
  → 3.1: PM content well presented and analysed?
  → 3.2: Well-organized references included?
  → 3.3: Evidence of engagement with others (staff/peers)?
  → grade_breadth, feedback_breadth, evidence_breadth
  ↓
CHECK 4: Link with Practice (30%)
  → 4.1: Evidence of link between study and PM practice?
  → 4.2: Link to own and/or others' practice?
  → grade_link, feedback_link, evidence_link
  ↓
╔══════════════════════════════════════════════════════════════════════════╗
║                    INTEGRITY & RESOLUTION NODE                           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ 1. EVIDENCE VALIDATION: Does evidence_* actually support feedback_*?     ║
║    → IF NO: Re-scan essay for a better quote. If none, lower the grade.  ║
║                                                                          ║
║ 2. GRADE CALIBRATION: Is the grade consistent with the feedback tone?    ║
║    → (e.g., If feedback says "Major gaps," grade cannot be > 50%).       ║
║    → IF CLASH: Re-examine rubric descriptors and adjust grade.           ║
║                                                                          ║
║ 3. LOGIC SYNC: If Reflection (Ch. 2) is "No Model," is 2.2A empty?       ║
║    → IF NO: Move data from 2.2A to 2.2B and re-evaluate style.           ║
║                                                                          ║
║ 4. HOLISTIC OVERLAY: Do the sub-scores match the overall "feel"?         ║
║    → IF CLASH: Re-evaluate the most "extreme" score (highest or lowest). ║
╚══════════════════════════════════════════════════════════════════════════╝
  ↓
FINAL GRADE CALCULATION
  → Weighted: 20% Pres + 20% Refl + 30% Breadth + 30% Link
  → Generate Consolidated Feedback Report
```

 ```text
 ┌──────────────────────────┐
 │          START           │
 └────────────┬─────────────┘
              ↓
 ┌──────────────────────────┐
 │  READ ESSAY + RUBRIC     │
 └────────────┬─────────────┘
              ↓
 ┌─────────────────────────────────────────────────┐
 │ CHECK 1: PRESENTATION (20%)                     │
 │ Assess overall communication and formatting     │
 │ → grade_pres, fb_pres, ev_pres                  │
 └────────────┬────────────────────────────────────┘
              ↓
 ┌─────────────────────────────────────────────────┐
 │ CHECK 2: REFLECTION (20%)                       │
 │ Check for evidence of continuous reflection     │
 └───────┬──────────────────────────┬──────────────┘
         │                          │
    [Used Model?]            [No Model?]
         ↓                          ↓
 ┌───────────────────────┐  ┌────────────────────────┐
 │ CHECK 2A:             │  │ CHECK 2B:              │
 │ Model Correctness &   │  │ Reflective language    │
 │ degree of application │  │ & writing style quality│
 └───────┬───────────────┘  └───────┬────────────────┘
         │                          │
         └────────────┬─────────────┘
                      ↓
 ┌─────────────────────────────────────────────────┐
 │ CHECK 3: BREADTH (30%)                          │
 │ 1. PM content presentation and analysis         │
 │ 2. Organization of references                   │
 │ 3. Engagement with others (staff/peers)         │
 │ → grade_breadth, fb_breadth, ev_breadth         │
 └────────────┬────────────────────────────────────┘
              ↓
 ┌─────────────────────────────────────────────────┐
 │ CHECK 4: LINK WITH PRACTICE (30%)               │
 │ Assess link between study and PM practice       │
 │ Look for evidence of links to:                  │
 │ • Student's own practice AND/OR                 │
 │ • Practice of others / wider profession         │
 │ → grade_link, fb_link, ev_link                  │
 └────────────┬────────────────────────────────────┘
              ↓
 ┌─────────────────────────────────────────────────┐
 │        INTEGRITY & RESOLUTION NODE              │
 │ 1. VALIDATE: Does quote (ev_*) support (fb_*)?  │
 │    IF NO: Re-scan essay. If none, lower grade.  │
 │ 2. CALIBRATE: Does tone match score? (e.g. 80%=)│
 │    IF NO: Adjust grade to match descriptors.    │
 │ 3. SYNC: If 'No Model', is model grade empty?   │
 │    IF NO: Transfer data to 2B Style check.      │
 └────────────┬────────────────────────────────────┘
              ↓
 ┌──────────────────────────┐
 │  FINAL MARK + FEEDBACK   │
 │   Aggregate weighted     │
 │   (20/20/30/30) scores   │
 └──────────────────────────┘
```

In [ ]:
#“Multi-pass rubric decomposition with aggregation”, and it addresses two common problems in single-pass
#grading: missed criteria and brittle reasoning, by scoring each criterion separately then combining.
import os
import json
from types import SimpleNamespace

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

from docx import Document

In [ ]:
# -------------------------------------------------------------------------
# Global prompts for marking
# -------------------------------------------------------------------------

SYSTEM_PROMPT = """You are an academic tutor marking a reflective essay.

Brief of the essay given to the students (follow strictly):
Write a reflective essay on your learning from this module. The essay should be written in the first person.
The essay will have two main areas of reflection:
- Reflect on learning
- Reflect on application

In writing the essay the student should balance personal perspective with good academic practice and rigorous thinking.

Typical questions include:
- What were the most important ideas you learned, and why?
- What was your learning process, and what methods helped?
- What went well and what went badly?
- What surprised you?
- How can you apply what you learned to your work?
- How did you engage with others to support your learning (students, colleagues, staff, online)?

Mark only from evidence in the essay. Do not invent details.
If evidence is weak or missing, score low.
"""

MARKING_SCHEME = """Marking rubric (weights sum to 100):
- Overall presentation and quality of communication 20%
- Evidence of continuous reflection during study 20%
- Breadth of study, including engagement with others (students and staff) 30%
- Linking studies with project management practice (own and general) 30%

Mark ranges:
0–49 = Fail (criteria not met or met at an inadequate level)
50–59 = Pass (criteria met at a basic level)
60–69 = Merit (criteria met well, with good understanding and analysis)
70+    = Distinction (criteria met at an excellent level, showing originality, critical insight, and clear argument)
"""

JSON_INSTRUCTIONS = """You must respond with a SINGLE JSON object and nothing else.

The JSON must have exactly these keys:
- "grade"    : an integer within the allowed range stated in the question
- "feedback" : 3 to 6 sentences as an academic tutor, specific and concrete
- "evidence" : one or two short quotes or precise references from the essay

Do NOT include any text before or after the JSON.
"""

INTEGRITY_JSON_INSTRUCTIONS = """
You are now performing an integrity and resolution check.
INTEGRITY CHECK (CRITICAL OUTPUT FORMAT).
You must output EXACTLY one JSON object and nothing else.
The FIRST character of your response must be { and the LAST character must be }.
No prose, no markdown, no code fences.

The JSON must have exactly these keys:
- "presentation": {"grade": int, "feedback": str, "evidence": str}
- "reflection"  : {"grade": int, "feedback": str, "evidence": str}
- "breadth"     : {"grade": int, "feedback": str, "evidence": str}
- "link"        : {"grade": int, "feedback": str, "evidence": str}
- "notes"       : a short string (optional notes for the marker, not for students)

RESOLUTION CHECK:
Resolution Rules:
1) EVIDENCE VALIDATION: If a quote does not support the grade and feedback, search the essay again. If no evidence exists, you MUST lower the grade.
2) TONE CALIBRATION: Ensure the grade matches the feedback tone (high praise needs a high mark, major gaps cannot have a high mark).
3) LOGIC RESOLUTION: If the reflection route is "no_model", do not mention model correctness or model stages in reflection feedback.
4) CONSISTENCY: If Breadth says no engagement with peers/staff, Link must not claim learning from discussions with peers/staff unless it quotes evidence. Link may still discuss the practice of others via reading, cases, and examples.

Keep grades within ranges:
- Presentation: 0..20
- Reflection: 0..20
- Breadth: 0..30
- Link with practice: 0..30
CRITICAL:Return JSON only.
"""

In [ ]:
# -------------------------------------------------------------------------
# 1. Local LLM
# -------------------------------------------------------------------------

#MODEL_DIR = "/path/to/project/models/deepseek-7b"
#MODEL_DIR = "/path/to/project/models/deepseek-qwen14b-4bit"
#MODEL_DIR = "/path/to/project/models/phi-4"
MODEL_DIR = "/path/to/project/models/qwen2.5-7b"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Has chat_template?", bool(getattr(tokenizer, "chat_template", None)))
print((tokenizer.chat_template or "")[:300])


def _invoke(messages, max_new_tokens=1024):
    chat_msgs = []
    for m in messages:
        role = "system" if isinstance(m, SystemMessage) else "user"
        chat_msgs.append({"role": role, "content": m.content})

    if getattr(tokenizer, "chat_template", None):
        prompt_text = tokenizer.apply_chat_template(
            chat_msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = ""
        for msg in chat_msgs:
            prefix = "System: " if msg["role"] == "system" else "User: "
            prompt_text += prefix + msg["content"] + "\n"

    out = text_pipeline(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.4,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        return_full_text=False,
    )

    return SimpleNamespace(content=out[0]["generated_text"].strip())


llm = SimpleNamespace(invoke=_invoke)


In [ ]:
# -------------------------------------------------------------------------
# 2. State
# -------------------------------------------------------------------------

class MarkingState(TypedDict, total=False):
    essay: str

    grade_pres: float
    fb_pres: str
    ev_pres: str

    model_used: bool
    grade_refl: float
    fb_refl: str
    ev_refl: str
    refl_route: str  # "model" or "no_model"

    grade_breadth: float
    fb_breadth: str
    ev_breadth: str

    grade_link: float
    fb_link: str
    ev_link: str

    integrity_notes: str

    final_grade: float
    final_feedback: str


In [ ]:
# -------------------------------------------------------------------------
# 3. Helpers
# -------------------------------------------------------------------------

def load_docx_text(path: str) -> str:
    doc = Document(path)
    paras = [p.text for p in doc.paragraphs]
    return "\n".join(p for p in paras if p.strip())


def _extract_json_block(text: str) -> dict:
    text = (text or "").strip()
    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in model output:\n{text[:400]}")
    depth = 0
    end = None
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end is None:
        raise ValueError(f"No balanced JSON object found in model output:\n{text[:400]}")
    return json.loads(text[start:end])


def _clamp_int(value, lo: int, hi: int) -> int:
    try:
        v = int(value)
    except Exception:
        v = lo
    if v < lo:
        return lo
    if v > hi:
        return hi
    return v


def run_check(state: MarkingState, instruction: str, min_grade: int, max_grade: int) -> dict:
    sys = SystemMessage(content=SYSTEM_PROMPT + "\n\n" + MARKING_SCHEME + "\n\n" + JSON_INSTRUCTIONS)
    human = HumanMessage(
        content=(
            f"Allowed grade range: {min_grade} to {max_grade} (integers only).\n\n"
            "Student essay:\n"
            f"{state['essay']}\n\n"
            f"Mark ONLY this aspect now:\n{instruction}\n\n"
            "Return ONLY the JSON object."
        )
    )

    response = llm.invoke([sys, human])
    raw = response.content if isinstance(response.content, str) else str(response.content)
    data = _extract_json_block(raw)

    data["grade"] = _clamp_int(data.get("grade", min_grade), min_grade, max_grade)
    data["feedback"] = str(data.get("feedback", "")).strip()
    data["evidence"] = str(data.get("evidence", "")).strip()
    return data


def run_integrity(state: MarkingState) -> dict:
    sys = SystemMessage(content=SYSTEM_PROMPT + "\n\n" + MARKING_SCHEME + "\n\n" + INTEGRITY_JSON_INSTRUCTIONS)
    human = HumanMessage(
        content=(
            "Student essay:\n"
            f"{state['essay']}\n\n"
            "Current section results:\n"
            f"Presentation: {int(state.get('grade_pres', 0))}/20\n"
            f"Feedback: {state.get('fb_pres', '')}\n"
            f"Evidence: {state.get('ev_pres', '')}\n\n"
            f"Reflection route: {state.get('refl_route', '')}\n"
            f"Reflection: {int(state.get('grade_refl', 0))}/20\n"
            f"Feedback: {state.get('fb_refl', '')}\n"
            f"Evidence: {state.get('ev_refl', '')}\n\n"
            f"Breadth: {int(state.get('grade_breadth', 0))}/30\n"
            f"Feedback: {state.get('fb_breadth', '')}\n"
            f"Evidence: {state.get('ev_breadth', '')}\n\n"
            f"Link: {int(state.get('grade_link', 0))}/30\n"
            f"Feedback: {state.get('fb_link', '')}\n"
            f"Evidence: {state.get('ev_link', '')}\n\n"
            "Apply the rules and return corrected values.\n"
            "Return JSON only.\n"
        )
    )

    response = llm.invoke([sys, human])
    raw = response.content if isinstance(response.content, str) else str(response.content)
    return _extract_json_block(raw)

In [ ]:
# -------------------------------------------------------------------------
# 4. Nodes (flowchart)
# -------------------------------------------------------------------------

def check_presentation(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 1 Presentation (20 points): assess overall communication quality, structure, formatting, clarity, flow, and academic tone.",
        0,
        20,
    )
    return {"grade_pres": data["grade"], "fb_pres": data["feedback"], "ev_pres": data["evidence"]}


def check_model_used(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 2 Reflection: does the student explicitly use a named conventional reflection model (Gibbs, Kolb, Schön, Rolfe, Borton, etc.)? "
        "Grade must be 1 if a named model is used, otherwise 0.",
        0,
        1,
    )
    return {"model_used": int(data["grade"]) == 1}


def route_reflection(state: MarkingState) -> str:
    return "check_reflection_model" if state.get("model_used") else "check_reflection_no_model"


def check_reflection_model(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 2A Reflection (20 points): a conventional model is used. Assess evidence of continuous reflection during study and the correctness and degree of model application. "
        "Reward depth, specificity, and reflection on learning and application.",
        0,
        20,
    )
    return {"grade_refl": data["grade"], "fb_refl": data["feedback"], "ev_refl": data["evidence"], "refl_route": "model"}


def check_reflection_no_model(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 2B Reflection (20 points): no model is used. Assess evidence of continuous reflection during study using reflective language and writing style quality. "
        "Reward depth, specificity, and reflection on learning and application.",
        0,
        20,
    )
    return {"grade_refl": data["grade"], "fb_refl": data["feedback"], "ev_refl": data["evidence"], "refl_route": "no_model"}


def check_breadth(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 3 Breadth (30 points): assess breadth of study across ALL of these: "
        "(1) project management content presentation and analysis, "
        "(2) organisation and use of references, "
        "(3) engagement with others (peers, staff, colleagues). "
        "Give a single holistic score out of 30 based on the overall breadth evidenced.",
        0,
        30,
    )
    return {"grade_breadth": data["grade"], "fb_breadth": data["feedback"], "ev_breadth": data["evidence"]}


def check_link(state: MarkingState) -> dict:
    data = run_check(
        state,
        "CHECK 4 Link with practice (30 points): evaluate the link between what was studied and project management practice. "
        "Practice can be the student's own practice and responsibilities and/or the practice of others and the wider profession (organisations, teams, industry practice, cases, standards). "
        "The student may address either route or both. Do not reduce the mark solely because only one route is addressed. "
        "Award marks based on depth, specificity, correctness, and evidence.",
        0,
        30,
    )
    return {"grade_link": data["grade"], "fb_link": data["feedback"], "ev_link": data["evidence"]}


def integrity_node(state: MarkingState) -> dict:
    data = run_integrity(state)

    pres = data.get("presentation", {})
    refl = data.get("reflection", {})
    br = data.get("breadth", {})
    lk = data.get("link", {})

    return {
        "grade_pres": _clamp_int(pres.get("grade", state.get("grade_pres", 0)), 0, 20),
        "fb_pres": str(pres.get("feedback", state.get("fb_pres", ""))).strip(),
        "ev_pres": str(pres.get("evidence", state.get("ev_pres", ""))).strip(),

        "grade_refl": _clamp_int(refl.get("grade", state.get("grade_refl", 0)), 0, 20),
        "fb_refl": str(refl.get("feedback", state.get("fb_refl", ""))).strip(),
        "ev_refl": str(refl.get("evidence", state.get("ev_refl", ""))).strip(),

        "grade_breadth": _clamp_int(br.get("grade", state.get("grade_breadth", 0)), 0, 30),
        "fb_breadth": str(br.get("feedback", state.get("fb_breadth", ""))).strip(),
        "ev_breadth": str(br.get("evidence", state.get("ev_breadth", ""))).strip(),

        "grade_link": _clamp_int(lk.get("grade", state.get("grade_link", 0)), 0, 30),
        "fb_link": str(lk.get("feedback", state.get("fb_link", ""))).strip(),
        "ev_link": str(lk.get("evidence", state.get("ev_link", ""))).strip(),

        "integrity_notes": str(data.get("notes", "")).strip(),
    }


def final_grade_node(state: MarkingState) -> dict:
    final_grade = int(state.get("grade_pres", 0)) + int(state.get("grade_refl", 0)) + int(state.get("grade_breadth", 0)) + int(state.get("grade_link", 0))
    final_grade = _clamp_int(final_grade, 0, 100)

    final_feedback = "\n".join(
        [
            f"Presentation: {int(state.get('grade_pres', 0))}/20. {state.get('fb_pres', '')}",
            f"Reflection: {int(state.get('grade_refl', 0))}/20. {state.get('fb_refl', '')}",
            f"Breadth: {int(state.get('grade_breadth', 0))}/30. {state.get('fb_breadth', '')}",
            f"Link with practice: {int(state.get('grade_link', 0))}/30. {state.get('fb_link', '')}",
        ]
    )

    return {"final_grade": final_grade, "final_feedback": final_feedback}


In [ ]:
# -------------------------------------------------------------------------
# 5. Workflow (same style as GRADER4)
# -------------------------------------------------------------------------

def build_workflow():
    graph = StateGraph(MarkingState)

    graph.add_node("check_presentation", check_presentation)
    graph.add_node("check_model_used", check_model_used)
    graph.add_node("check_reflection_model", check_reflection_model)
    graph.add_node("check_reflection_no_model", check_reflection_no_model)
    graph.add_node("check_breadth", check_breadth)
    graph.add_node("check_link", check_link)
    graph.add_node("integrity_node", integrity_node)
    graph.add_node("final_grade_node", final_grade_node)

    graph.set_entry_point("check_presentation")

    graph.add_edge("check_presentation", "check_model_used")

    graph.add_conditional_edges(
        "check_model_used",
        route_reflection,
        {
            "check_reflection_model": "check_reflection_model",
            "check_reflection_no_model": "check_reflection_no_model",
        },
    )

    graph.add_edge("check_reflection_model", "check_breadth")
    graph.add_edge("check_reflection_no_model", "check_breadth")
    graph.add_edge("check_breadth", "check_link")
    graph.add_edge("check_link", "integrity_node")
    graph.add_edge("integrity_node", "final_grade_node")
    graph.add_edge("final_grade_node", END)

    return graph.compile()


In [ ]:
# -------------------------------------------------------------------------
# 6. Run
# -------------------------------------------------------------------------

ESSAY_DIR = "/path/to/project/essays"
MAX_RETRIES = 6

if __name__ == "__main__":
    app = build_workflow()

    for name in sorted(os.listdir(ESSAY_DIR)):
        if not name.lower().endswith(".docx"):
            continue

        path = os.path.join(ESSAY_DIR, name)
        essay_text = load_docx_text(path)

        result = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                result = app.invoke({"essay": essay_text})
                break
            except Exception as e:
                print(f"Attempt {attempt}/{MAX_RETRIES} for {name} failed: {e}")

        if result is None:
            print(f"{name} SKIPPED after {MAX_RETRIES} attempts.\n")
            continue

        print(f"\n--- BEGIN {name} ---")
        print("FINAL GRADE:", result.get("final_grade"))
        if result.get("integrity_notes"):
            print("INTEGRITY NOTES:", result.get("integrity_notes"))
        print("\nFEEDBACK:\n", result.get("final_feedback", ""))
        print(f"--- END {name} ---")
